## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**: 李菲


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code here

#include <bits/stdc++.h>
using namespace std;

struct Op {
    int type, x; // 0 swap, 1 xor, 2 add
};

struct Key {
    unsigned long long a, b, c;
    bool operator==(const Key& other) const {
        return a == other.a && b == other.b && c == other.c;
    }
};

struct KeyHash {
    size_t operator()(const Key& k) const {
        return (size_t)(k.a * 1000003ULL ^ k.b * 10007ULL ^ k.c);
    }
};

int N, A, B, M;
vector<int> arrv, posv, originalArr;
vector<Op> answer;

vector<Op> allOps;
vector<signed char> distPair;
vector<int> prePair;
vector<Op> preOpPair;

int lowbit(int x) {
    return x & -x;
}

int applyValue(int v, Op op, int mod) {
    if (op.type == 1) return v ^ op.x;
    if (op.type == 2) return (v + op.x) & (mod - 1);
    return v;
}

Op inverseOp(Op op) {
    if (op.type == 1) return op;
    if (op.type == 2) return {2, (N - op.x) & (N - 1)};
    return op;
}

void addAnswer(Op op) {
    if ((op.type == 1 || op.type == 2) && op.x == 0) return;
    answer.push_back(op);
}

int pairKey(int x, int y) {
    return x * N + y;
}

void putBits(Key& k, int bit, unsigned long long v) {
    for (int i = 0; i < 5; i++) {
        if ((v >> i) & 1ULL) {
            int p = bit + i;
            if (p < 64) k.a |= 1ULL << p;
            else if (p < 128) k.b |= 1ULL << (p - 64);
            else k.c |= 1ULL << (p - 128);
        }
    }
}

Key encodePerm(const array<unsigned char, 32>& p, int m) {
    Key k{0, 0, 0};

    for (int i = 0; i < m; i++) {
        putBits(k, i * 5, p[i]);
    }

    return k;
}

bool sameTarget(const array<unsigned char, 32>& p, const vector<int>& target, int m) {
    for (int i = 0; i < m; i++) {
        if ((int)p[i] != target[i]) return false;
    }
    return true;
}

bool bfsResidue(int m, const vector<int>& target, vector<Op>& seq) {
    seq.clear();

    if (m == 1) return true;

    array<unsigned char, 32> start{};
    for (int i = 0; i < m; i++) start[i] = i;

    if (sameTarget(start, target, m)) return true;

    vector<array<unsigned char, 32>> states;
    vector<int> pre;
    vector<Op> preop;

    unordered_map<Key, int, KeyHash> id;
    id.reserve(1200000);

    states.push_back(start);
    pre.push_back(-1);
    preop.push_back({0, 0});
    id[encodePerm(start, m)] = 0;

    queue<int> q;
    q.push(0);

    while (!q.empty()) {
        int u = q.front();
        q.pop();

        for (int type = 1; type <= 2; type++) {
            for (int x = 1; x < m; x++) {
                array<unsigned char, 32> nxt = states[u];

                for (int i = 0; i < m; i++) {
                    if (type == 1) nxt[i] ^= x;
                    else nxt[i] = (nxt[i] + x) & (m - 1);
                }

                Key key = encodePerm(nxt, m);

                if (id.count(key)) continue;

                int nid = (int)states.size();
                id[key] = nid;

                states.push_back(nxt);
                pre.push_back(u);
                preop.push_back({type, x});

                if (sameTarget(nxt, target, m)) {
                    vector<Op> rev;
                    int cur = nid;

                    while (pre[cur] != -1) {
                        rev.push_back(preop[cur]);
                        cur = pre[cur];
                    }

                    reverse(rev.begin(), rev.end());
                    seq = rev;
                    return true;
                }

                q.push(nid);
            }
        }
    }

    return false;
}

bool checkSeq(const vector<int>& target, const vector<Op>& seq, int m) {
    for (int i = 0; i < m; i++) {
        int v = i;

        for (Op op : seq) {
            if (op.type == 1) v ^= op.x;
            else v = (v + op.x) & (m - 1);
        }

        if (v != target[i]) return false;
    }

    return true;
}

void appendOp(vector<Op>& seq, Op op, int mod) {
    op.x &= mod - 1;
    if ((op.type == 1 || op.type == 2) && op.x == 0) return;
    seq.push_back(op);
}

void appendKernelOps(int m, const vector<int>& h, vector<Op>& seq) {
    int half = m >> 1;

    if (m == 2) {
        if (h[0]) appendOp(seq, {1, 1}, m);
        return;
    }

    int q = m >> 2;
    int c = h[0] ^ h[q];
    vector<int> f(q);

    for (int i = 0; i < q; i++) {
        f[i] = h[i];
        if ((h[i] ^ h[i + q]) != c) {
            seq.clear();
            seq.push_back({-1, -1});
            return;
        }
    }

    int ones = 0;
    for (int v : f) ones += v;

    if (ones > q / 2) {
        appendOp(seq, {1, half}, m);
        for (int& v : f) v ^= 1;
    }

    for (int t = 0; t < q; t++) {
        if (!f[t]) continue;

        int mask = t ^ (q - 1);
        appendOp(seq, {1, mask}, m);
        appendOp(seq, {2, 1}, m);
        appendOp(seq, {1, q}, m);
        appendOp(seq, {2, m - 1}, m);
        appendOp(seq, {1, q}, m);
        appendOp(seq, {1, mask}, m);
    }

    if (c) {
        appendOp(seq, {1, q}, m);
        appendOp(seq, {2, m - q}, m);
    }
}

vector<int> buildPermutationByOps(int m, const vector<Op>& seq) {
    vector<int> p(m);
    iota(p.begin(), p.end(), 0);

    for (Op op : seq) {
        for (int& v : p) {
            v = applyValue(v, op, m);
        }
    }

    return p;
}

bool solveResidueExact(int m, const vector<int>& target, vector<Op>& seq) {
    seq.clear();

    if (m == 1) return true;

    int half = m >> 1;
    vector<int> lower(half);

    for (int i = 0; i < half; i++) {
        int a = target[i] & (half - 1);
        int b = target[i + half] & (half - 1);

        if (a != b) return false;
        lower[i] = a;
    }

    vector<Op> lowSeq;
    if (!solveResidueExact(half, lower, lowSeq)) return false;

    vector<int> lowPerm = buildPermutationByOps(m, lowSeq);
    vector<int> invLow(m);
    for (int i = 0; i < m; i++) {
        invLow[lowPerm[i]] = i;
    }

    vector<int> h(half, -1);

    for (int z = 0; z < m; z++) {
        int original = invLow[z];
        int r = target[original];

        if ((r & (half - 1)) != (z & (half - 1))) return false;

        int u = z & (half - 1);
        int bit = ((r ^ z) & half) ? 1 : 0;

        if (h[u] == -1) h[u] = bit;
        else if (h[u] != bit) return false;
    }

    vector<Op> kernelSeq;
    appendKernelOps(m, h, kernelSeq);

    if (!kernelSeq.empty() && kernelSeq[0].type == -1) return false;

    seq = lowSeq;
    seq.insert(seq.end(), kernelSeq.begin(), kernelSeq.end());

    return checkSeq(target, seq, m);
}

bool findPreOps(int m, const vector<int>& target, vector<Op>& seq) {
    seq.clear();

    if (m == 1) return true;

    return solveResidueExact(m, target, seq);
}

void buildPairTable() {
    int total = N * N;

    distPair.assign(total, -1);
    prePair.assign(total, -1);
    preOpPair.assign(total, {0, 0});

    vector<int> cur, nxt;

    int k1 = pairKey(A, B);
    int k2 = pairKey(B, A);

    distPair[k1] = 0;
    cur.push_back(k1);

    if (k2 != k1) {
        distPair[k2] = 0;
        cur.push_back(k2);
    }

    for (int dep = 0; dep < 3; dep++) {
        nxt.clear();

        for (int key : cur) {
            int x = key / N;
            int y = key % N;

            for (Op op : allOps) {
                int nx = applyValue(x, op, N);
                int ny = applyValue(y, op, N);

                int nk = pairKey(nx, ny);

                if (distPair[nk] != -1) continue;

                distPair[nk] = dep + 1;
                prePair[nk] = key;
                preOpPair[nk] = op;

                nxt.push_back(nk);
            }
        }

        cur.swap(nxt);
    }
}

bool getSeqToAB(int x, int y, vector<Op>& seq) {
    seq.clear();

    int key = pairKey(x, y);

    if (distPair[key] == -1) return false;

    vector<Op> forward;

    while (prePair[key] != -1) {
        forward.push_back(preOpPair[key]);
        key = prePair[key];
    }

    // forward is collected from child to parent. To go from child back to
    // (A,B), apply the inverse operations in this same order.
    for (Op op : forward) {
        seq.push_back(inverseOp(op));
    }

    return true;
}

bool realSwapByEdge(int x, int y) {
    if (x == y) return true;

    vector<Op> seq;

    if (!getSeqToAB(x, y, seq)) return false;

    for (Op op : seq) addAnswer(op);

    addAnswer({0, 0});

    for (int i = (int)seq.size() - 1; i >= 0; i--) {
        addAnswer(inverseOp(seq[i]));
    }

    int px = posv[x];
    int py = posv[y];

    swap(arrv[px], arrv[py]);
    swap(posv[x], posv[y]);

    return true;
}

bool swapSameClass(int x, int y) {
    if (x == y) return true;

    if (lowbit(x ^ y) == M) {
        return realSwapByEdge(x, y);
    }

    int z = x ^ M;

    if (!realSwapByEdge(x, z)) return false;
    if (!realSwapByEdge(y, z)) return false;
    if (!realSwapByEdge(x, z)) return false;

    return true;
}

bool verifyAnswer() {
    vector<int> cur = originalArr;

    for (Op op : answer) {
        if (op.type == 0) {
            int pa = -1, pb = -1;

            for (int i = 0; i < N; i++) {
                if (cur[i] == A) pa = i;
                if (cur[i] == B) pb = i;
            }

            if (pa == -1 || pb == -1) return false;

            swap(cur[pa], cur[pb]);
        } else {
            for (int i = 0; i < N; i++) {
                cur[i] = applyValue(cur[i], op, N);
            }
        }
    }

    for (int i = 0; i < N; i++) {
        if (cur[i] != i) return false;
    }

    return true;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    cin >> N;
    cin >> A >> B;

    arrv.resize(N);
    posv.resize(N);
    originalArr.resize(N);

    for (int i = 0; i < N; i++) {
        cin >> arrv[i];
        originalArr[i] = arrv[i];
        posv[arrv[i]] = i;
    }

    M = lowbit(A ^ B);

    for (int x = 1; x < N; x++) {
        allOps.push_back({1, x});
        allOps.push_back({2, x});
    }

    vector<int> need(M, -1);

    for (int i = 0; i < N; i++) {
        int r = arrv[i] & (M - 1);
        int s = i & (M - 1);

        if (need[r] == -1) need[r] = s;
        else if (need[r] != s) {
            cout << -1 << '\n';
            return 0;
        }
    }

    for (int i = 0; i < M; i++) {
        if (need[i] == -1) {
            cout << -1 << '\n';
            return 0;
        }
    }

    vector<Op> preOps;

    if (!findPreOps(M, need, preOps)) {
        cout << -1 << '\n';
        return 0;
    }

    for (Op op : preOps) {
        addAnswer(op);

        for (int i = 0; i < N; i++) {
            arrv[i] = applyValue(arrv[i], op, N);
        }
    }

    for (int i = 0; i < N; i++) {
        posv[arrv[i]] = i;
    }

    for (int i = 0; i < N; i++) {
        if ((arrv[i] & (M - 1)) != (i & (M - 1))) {
            cout << -1 << '\n';
            return 0;
        }
    }

    buildPairTable();

    for (int i = 0; i < N; i++) {
        if (arrv[i] == i) continue;

        int x = arrv[i];
        int y = i;

        if ((x & (M - 1)) != (y & (M - 1))) {
            cout << -1 << '\n';
            return 0;
        }

        if (!swapSameClass(x, y)) {
            cout << -1 << '\n';
            return 0;
        }

        if ((int)answer.size() > 32768) {
            cout << -1 << '\n';
            return 0;
        }
    }

    if ((int)answer.size() > 32768 || !verifyAnswer()) {
        cout << -1 << '\n';
        return 0;
    }

    cout << answer.size() << '\n';

    for (Op op : answer) {
        if (op.type == 0) {
            cout << 0 << '\n';
        } else {
            cout << op.type << ' ' << op.x << '\n';
        }
    }

    return 0;
}


## B 长跑

In [ ]:
## add your code here

#include <bits/stdc++.h>
using namespace std;
 
int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
 
    int N, L, Maxn, S;
 
    while (cin >> N >> L >> Maxn >> S) {
        map<int, int> mp;
 
        for (int i = 0; i < N; i++) {
            int P, C;
            cin >> P >> C;
 
            if (P <= 0 || P >= L) continue;
 
            if (!mp.count(P)) mp[P] = C;
            else mp[P] = min(mp[P], C);
        }
 
        vector<int> pos;
        vector<int> cost;
 
        pos.push_back(0);
        cost.push_back(0);
 
        for (auto &it : mp) {
            pos.push_back(it.first);
            cost.push_back(it.second);
        }
 
        pos.push_back(L);
        cost.push_back(0);
 
        int m = pos.size();
        const long long INF = 1e18;
 
        vector<long long> dp(m, INF);
        dp[0] = 0;
 
        for (int i = 0; i < m; i++) {
            if (dp[i] > S) continue;
 
            for (int j = i + 1; j < m; j++) {
                if (pos[j] - pos[i] > Maxn) break;
 
                if (j == m - 1) {
                    dp[j] = min(dp[j], dp[i]);
                } else {
                    dp[j] = min(dp[j], dp[i] + cost[j]);
                }
            }
        }
 
        if (dp[m - 1] <= S) {
            cout << "Yes\n";
        } else {
            cout << "No\n";
        }
    }
 
    return 0;
}

## C 最长回文

In [ ]:
## add your code here

#include <bits/stdc++.h>
using namespace std;
 
using ll = long long;
 
const ll MOD1 = 1000000007;
const ll MOD2 = 1000000009;
const ll BASE = 131;
 
struct Hash {
    vector<ll> h1, h2, p1, p2;
 
    Hash() {}
 
    Hash(const string &s) {
        int n = s.size();
        h1.assign(n + 1, 0);
        h2.assign(n + 1, 0);
        p1.assign(n + 1, 1);
        p2.assign(n + 1, 1);
 
        for (int i = 0; i < n; i++) {
            p1[i + 1] = p1[i] * BASE % MOD1;
            p2[i + 1] = p2[i] * BASE % MOD2;
 
            h1[i + 1] = (h1[i] * BASE + s[i]) % MOD1;
            h2[i + 1] = (h2[i] * BASE + s[i]) % MOD2;
        }
    }
 
    pair<ll, ll> get(int l, int len) {
        if (len <= 0) return {0, 0};
 
        ll x1 = (h1[l + len] - h1[l] * p1[len]) % MOD1;
        ll x2 = (h2[l + len] - h2[l] * p2[len]) % MOD2;
 
        if (x1 < 0) x1 += MOD1;
        if (x2 < 0) x2 += MOD2;
 
        return {x1, x2};
    }
};
 
pair<vector<int>, vector<int>> manacher(const string &s) {
    int n = s.size();
 
    vector<int> d1(n), d2(n);
 
    for (int i = 0, l = 0, r = -1; i < n; i++) {
        int k = 1;
        if (i <= r) {
            k = min(d1[l + r - i], r - i + 1);
        }
 
        while (i - k >= 0 && i + k < n && s[i - k] == s[i + k]) {
            k++;
        }
 
        d1[i] = k;
 
        if (i + k - 1 > r) {
            l = i - k + 1;
            r = i + k - 1;
        }
    }
 
    for (int i = 0, l = 0, r = -1; i < n; i++) {
        int k = 0;
        if (i <= r) {
            k = min(d2[l + r - i + 1], r - i + 1);
        }
 
        while (i - k - 1 >= 0 && i + k < n && s[i - k - 1] == s[i + k]) {
            k++;
        }
 
        d2[i] = k;
 
        if (i + k - 1 > r) {
            l = i - k;
            r = i + k - 1;
        }
    }
 
    return {d1, d2};
}
 
int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
 
    int n;
    string A, B;
 
    cin >> n >> A >> B;
 
    string RA = A;
    reverse(RA.begin(), RA.end());
 
    Hash hRA(RA), hB(B);
 
    auto same = [&](int posRA, int posB, int len) {
        return hRA.get(posRA, len) == hB.get(posB, len);
    };
 
    auto lce = [&](int leftA, int rightB) {
        if (leftA < 0 || rightB >= n) return 0;
 
        int limit = min(leftA + 1, n - rightB);
        int posRA = n - 1 - leftA;
 
        int l = 0, r = limit;
        while (l < r) {
            int mid = (l + r + 1) / 2;
 
            if (same(posRA, rightB, mid)) {
                l = mid;
            } else {
                r = mid - 1;
            }
        }
 
        return l;
    };
 
    auto [d1A, d2A] = manacher(A);
    auto [d1B, d2B] = manacher(B);
 
    ll ans = 0;
 
    // 只选 A 中的回文串
    for (int i = 0; i < n; i++) {
        ans = max(ans, 2LL * d1A[i] - 1);
        ans = max(ans, 2LL * d2A[i]);
    }
 
    // 只选 B 中的回文串
    for (int i = 0; i < n; i++) {
        ans = max(ans, 2LL * d1B[i] - 1);
        ans = max(ans, 2LL * d2B[i]);
    }
 
    // 情况 1：中间回文在 A 中，奇数长度
    for (int c = 0; c < n; c++) {
        int R = d1A[c];
 
        int leftA = c - R;
        int rightB = c + R - 1;
 
        int k = lce(leftA, rightB);
 
        ans = max(ans, 2LL * R - 1 + 2LL * k);
    }
 
    // 情况 2：中间回文在 A 中，偶数长度
    // c 表示 A[c] 后面的缝
    for (int c = 0; c < n; c++) {
        int R = 0;
        if (c + 1 < n) R = d2A[c + 1];
 
        int leftA = c - R;
        int rightB = c + R;
 
        int k = lce(leftA, rightB);
 
        ans = max(ans, 2LL * R + 2LL * k);
    }
 
    // 情况 3：中间回文在 B 中，奇数长度
    for (int c = 0; c < n; c++) {
        int R = d1B[c];
 
        int startB = c - R + 1;
        int rightB = c + R;
 
        int k = lce(startB, rightB);
 
        ans = max(ans, 2LL * R - 1 + 2LL * k);
    }
 
    // 情况 4：中间回文在 B 中，偶数长度
    for (int c = 0; c + 1 < n; c++) {
        int R = d2B[c + 1];
 
        int startB = c - R + 1;
        int rightB = c + R + 1;
 
        int k = lce(startB, rightB);
 
        ans = max(ans, 2LL * R + 2LL * k);
    }
 
    cout << ans << '\n';
 
    return 0;
}

## D 优惠券

In [ ]:
## add your code here

#include <bits/stdc++.h>
using namespace std;
 
const int MAXX = 100000;
 
struct Fenwick {
    int n;
    vector<int> tree;
 
    Fenwick(int n = 0) {
        init(n);
    }
 
    void init(int n_) {
        n = n_;
        tree.assign(n + 2, 0);
    }
 
    void add(int idx, int val) {
        for (; idx <= n; idx += idx & -idx) {
            tree[idx] += val;
        }
    }
 
    int sum(int idx) {
        int res = 0;
        for (; idx > 0; idx -= idx & -idx) {
            res += tree[idx];
        }
        return res;
    }
 
    int total() {
        return sum(n);
    }
 
    // 找第 k 个可用问号的位置
    int kth(int k) {
        int idx = 0;
        int bit = 1;
        while ((bit << 1) <= n) bit <<= 1;
 
        for (; bit; bit >>= 1) {
            int nxt = idx + bit;
            if (nxt <= n && tree[nxt] < k) {
                idx = nxt;
                k -= tree[nxt];
            }
        }
 
        return idx + 1;
    }
 
    // 找第一个位置 > pos 的可用问号
    int firstAfter(int pos) {
        int before = sum(pos);
        if (total() == before) return -1;
        return kth(before + 1);
    }
};
 
int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
 
    int m;
 
    static int cnt[MAXX + 5];
    static int lastPos[MAXX + 5];
    static int vis[MAXX + 5];
 
    int tag = 1;
 
    while (cin >> m) {
        Fenwick bit(m);
 
        int ans = -1;
 
        for (int i = 1; i <= m; i++) {
            string op;
            cin >> op;
 
            if (op == "?") {
                if (ans == -1) {
                    bit.add(i, 1);
                }
                continue;
            }
 
            int x;
            cin >> x;
 
            if (vis[x] != tag) {
                vis[x] = tag;
                cnt[x] = 0;
                lastPos[x] = 0;
            }
 
            if (ans != -1) continue;
 
            if (op == "I") {
                cnt[x]++;
            } else {
                cnt[x]--;
            }
 
            if (cnt[x] < 0 || cnt[x] > 1) {
                int p = bit.firstAfter(lastPos[x]);
 
                if (p == -1) {
                    ans = i;
                } else {
                    bit.add(p, -1);
 
                    if (cnt[x] < 0) {
                        cnt[x] = 0;
                    } else {
                        cnt[x] = 1;
                    }
                }
            }
 
            lastPos[x] = i;
        }
 
        cout << ans << '\n';
 
        tag++;
    }
 
    return 0;
}

## E 任意点

In [ ]:
## add your code here

#include <bits/stdc++.h>
using namespace std;
 
struct DSU {
    vector<int> fa;
 
    DSU(int n = 0) {
        fa.resize(n);
        iota(fa.begin(), fa.end(), 0);
    }
 
    int find(int x) {
        if (fa[x] == x) return x;
        return fa[x] = find(fa[x]);
    }
 
    void unite(int a, int b) {
        int ra = find(a);
        int rb = find(b);
        if (ra != rb) fa[ra] = rb;
    }
};
 
int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
 
    int n;
    cin >> n;
 
    DSU dsu(n);
 
    map<int, int> sameX;
    map<int, int> sameY;
 
    for (int i = 0; i < n; i++) {
        int x, y;
        cin >> x >> y;
 
        if (sameX.count(x)) {
            dsu.unite(i, sameX[x]);
        } else {
            sameX[x] = i;
        }
 
        if (sameY.count(y)) {
            dsu.unite(i, sameY[y]);
        } else {
            sameY[y] = i;
        }
    }
 
    set<int> comp;
    for (int i = 0; i < n; i++) {
        comp.insert(dsu.find(i));
    }
 
    cout << (int)comp.size() - 1 << '\n';
 
    return 0;
}

## F 通配符匹配

In [ ]:
## add your code here

#include <bits/stdc++.h>
using namespace std;
 
struct PieceRef {
    int offset;
    int id;
};
 
struct Block {
    int len;
    vector<PieceRef> pieces;
};
 
vector<int> getPi(const string &p) {
    int n = p.size();
    vector<int> pi(n, 0);
 
    for (int i = 1; i < n; i++) {
        int j = pi[i - 1];
 
        while (j > 0 && p[i] != p[j]) {
            j = pi[j - 1];
        }
 
        if (p[i] == p[j]) {
            j++;
        }
 
        pi[i] = j;
    }
 
    return pi;
}
 
vector<unsigned char> kmpOccur(const string &s, const string &p) {
    int n = s.size();
    int m = p.size();
 
    vector<unsigned char> occ(n + 1, 0);
 
    if (m == 0 || m > n) return occ;
 
    vector<int> pi = getPi(p);
 
    int j = 0;
 
    for (int i = 0; i < n; i++) {
        while (j > 0 && s[i] != p[j]) {
            j = pi[j - 1];
        }
 
        if (s[i] == p[j]) {
            j++;
        }
 
        if (j == m) {
            int start = i - m + 1;
            occ[start] = 1;
            j = pi[j - 1];
        }
    }
 
    return occ;
}
 
bool blockMatchAt(const Block &b, int start, int n,
                  const vector<vector<unsigned char>> &occ) {
    if (start < 0 || start + b.len > n) return false;
 
    for (auto &p : b.pieces) {
        if (!occ[p.id][start + p.offset]) {
            return false;
        }
    }
 
    return true;
}
 
int findBlock(const Block &b, int left, int rightLimit,
              int n, const vector<vector<unsigned char>> &occ) {
    int maxStart = rightLimit - b.len;
 
    if (left > maxStart) return -1;
 
    if (b.pieces.empty()) {
        return left;
    }
 
    for (int st = left; st <= maxStart; st++) {
        bool ok = true;
 
        for (auto &p : b.pieces) {
            if (!occ[p.id][st + p.offset]) {
                ok = false;
                break;
            }
        }
 
        if (ok) return st;
    }
 
    return -1;
}
 
int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
 
    string pattern;
    cin >> pattern;
 
    int n;
    cin >> n;
 
    vector<string> files(n);
 
    for (int i = 0; i < n; i++) {
        cin >> files[i];
    }
 
    vector<string> pieces;
    vector<Block> blocks;
 
    int start = 0;
    int m = pattern.size();
 
    for (int i = 0; i <= m; i++) {
        if (i == m || pattern[i] == '*') {
            string part = pattern.substr(start, i - start);
 
            Block b;
            b.len = part.size();
 
            int last = 0;
 
            for (int j = 0; j <= (int)part.size(); j++) {
                if (j == (int)part.size() || part[j] == '?') {
                    if (j > last) {
                        string literal = part.substr(last, j - last);
                        int id = pieces.size();
                        pieces.push_back(literal);
                        b.pieces.push_back({last, id});
                    }
 
                    last = j + 1;
                }
            }
 
            blocks.push_back(b);
            start = i + 1;
        }
    }
 
    bool hasStar = false;
    for (char c : pattern) {
        if (c == '*') {
            hasStar = true;
            break;
        }
    }
 
    for (string &s : files) {
        int len = s.size();
 
        vector<vector<unsigned char>> occ;
 
        for (string &p : pieces) {
            occ.push_back(kmpOccur(s, p));
        }
 
        bool ok = true;
 
        if (!hasStar) {
            if (len != blocks[0].len) {
                ok = false;
            } else {
                ok = blockMatchAt(blocks[0], 0, len, occ);
            }
        } else {
            int first = 0;
            int last = (int)blocks.size() - 1;
 
            if (!blockMatchAt(blocks[first], 0, len, occ)) {
                ok = false;
            }
 
            int cur = blocks[first].len;
 
            int suffixStart = len - blocks[last].len;
 
            if (ok && suffixStart < 0) {
                ok = false;
            }
 
            if (ok && !blockMatchAt(blocks[last], suffixStart, len, occ)) {
                ok = false;
            }
 
            if (ok && cur > suffixStart) {
                ok = false;
            }
 
            for (int i = 1; ok && i < last; i++) {
                int pos = findBlock(blocks[i], cur, suffixStart, len, occ);
 
                if (pos == -1) {
                    ok = false;
                } else {
                    cur = pos + blocks[i].len;
                }
            }
 
            if (ok && cur > suffixStart) {
                ok = false;
            }
        }
 
        cout << (ok ? "YES" : "NO") << '\n';
    }
 
    return 0;
}

## G 汉诺塔

In [ ]:
## add your code here

#include <bits/stdc++.h>
using namespace std;
 
using ll = long long;
 
ll simulateSmall(int n, const vector<string>& order) {
    vector<int> st[3];
 
    for (int i = n; i >= 1; i--) {
        st[0].push_back(i);
    }
 
    int last = -1;
    ll step = 0;
 
    while (true) {
        if ((int)st[1].size() == n || (int)st[2].size() == n) {
            return step;
        }
 
        for (string op : order) {
            int from = op[0] - 'A';
            int to = op[1] - 'A';
 
            if (st[from].empty()) continue;
 
            int disk = st[from].back();
 
            if (disk == last) continue;
 
            if (!st[to].empty() && st[to].back() < disk) continue;
 
            st[from].pop_back();
            st[to].push_back(disk);
 
            last = disk;
            step++;
            break;
        }
    }
}
 
int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
 
    int n;
    cin >> n;
 
    vector<string> order(6);
 
    for (int i = 0; i < 6; i++) {
        cin >> order[i];
    }
 
    ll f2 = simulateSmall(2, order);
    ll f3 = simulateSmall(3, order);
 
    ll ans = 1;
 
    if (f2 == 5) {
        // 1, 5, 17, 53...
        for (int i = 2; i <= n; i++) {
            ans = ans * 3 + 2;
        }
    } else if (f3 == 7) {
        // 1, 3, 7, 15...
        for (int i = 2; i <= n; i++) {
            ans = ans * 2 + 1;
        }
    } else {
        // 1, 3, 9, 27...
        for (int i = 2; i <= n; i++) {
            ans = ans * 3;
        }
    }
 
    cout << ans << '\n';
 
    return 0;
}

## H 马步距离

In [ ]:
## add your code here

#include <bits/stdc++.h>
using namespace std;
 
using ll = long long;
 
ll knightDistance(ll x1, ll y1, ll x2, ll y2) {
    ll x = llabs(x1 - x2);
    ll y = llabs(y1 - y2);
 
    if (x < y) swap(x, y); // 保证 x >= y
 
    if (x == 0 && y == 0) return 0;
    if (x == 1 && y == 0) return 3;
    if (x == 2 && y == 2) return 4;
 
    ll ans = max((x + 1) / 2, (x + y + 2) / 3);
 
    if ((ans % 2) != ((x + y) % 2)) {
        ans++;
    }
 
    return ans;
}
 
int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
 
    ll xp, yp, xs, ys;
    cin >> xp >> yp >> xs >> ys;
 
    cout << knightDistance(xp, yp, xs, ys) << '\n';
 
    return 0;
}

## I 直方图最大矩形

In [ ]:
## add your code here

class Solution {
public:
    int largestRectangleArea(vector<int>& heights) {
        int n = heights.size();
        if (n == 0) return 0;
 
        stack<int> st;
        long long ans = 0;
 
        for (int i = 0; i <= n; i++) {
            int curHeight = (i == n ? 0 : heights[i]);
 
            while (!st.empty() && heights[st.top()] > curHeight) {
                int h = heights[st.top()];
                st.pop();
 
                int left = st.empty() ? -1 : st.top();
                int width = i - left - 1;
 
                ans = max(ans, 1LL * h * width);
            }
 
            st.push(i);
        }
 
        return (int)ans;
    }
};

## J 消防局的设立

In [ ]:
## add your code here

#include <bits/stdc++.h>
using namespace std;
 
const int MAXN = 10005;
 
int n;
vector<int> g[MAXN];
int parentNode[MAXN];
int depthNode[MAXN];
bool covered[MAXN];
 
void dfs(int u, int fa) {
    parentNode[u] = fa;
    depthNode[u] = depthNode[fa] + 1;
 
    for (int v : g[u]) {
        if (v == fa) continue;
        dfs(v, u);
    }
}
 
void cover(int u) {
    if (u <= 0) return;
 
    covered[u] = true;
 
    for (int v : g[u]) {
        covered[v] = true;
 
        for (int w : g[v]) {
            covered[w] = true;
        }
    }
}
 
int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
 
    cin >> n;
 
    for (int i = 2; i <= n; i++) {
        int a;
        cin >> a;
 
        g[i].push_back(a);
        g[a].push_back(i);
    }
 
    depthNode[0] = -1;
    dfs(1, 0);
 
    vector<int> nodes;
    for (int i = 1; i <= n; i++) {
        nodes.push_back(i);
    }
 
    sort(nodes.begin(), nodes.end(), [](int a, int b) {
        return depthNode[a] > depthNode[b];
    });
 
    int ans = 0;
 
    for (int u : nodes) {
        if (!covered[u]) {
            int p = parentNode[u];
            int pp = parentNode[p];
 
            int station;
            if (pp != 0) station = pp;
            else if (p != 0) station = p;
            else station = u;
 
            ans++;
            cover(station);
        }
    }
 
    cout << ans << '\n';
 
    return 0;
}